# Set Up Databricks Vector Search Index

Creates a **Delta Sync** Vector Search index backed by `main.scholarships.scheme_corpus`.
Embeddings are managed server-side by `databricks-gte-large-en` — no local ML model needed.

**Prerequisites:**
- Run `scholarship_ingest.ipynb` first (writes the Delta table with CDF enabled).
- A Databricks cluster / serverless SQL with Vector Search enabled in the workspace.

**Run order:**
1. `scholarship_ingest.ipynb`
2. **This notebook** (once — safe to re-run; endpoint/index creation is idempotent)

**Output:** VS endpoint `scholarship-vs-endpoint` + index `main.scholarships.scheme_vs_index`

In [ ]:
CATALOG        = "main"
SCHEMA         = "scholarships"
TABLE          = "scheme_corpus"
VS_ENDPOINT    = "scholarship-vs-endpoint"
VS_INDEX       = f"{CATALOG}.{SCHEMA}.scheme_vs_index"
EMBED_ENDPOINT = "databricks-gte-large-en"   # Foundation Model already available in workspace

print(f"Source table : {CATALOG}.{SCHEMA}.{TABLE}")
print(f"VS endpoint  : {VS_ENDPOINT}")
print(f"VS index     : {VS_INDEX}")
print(f"Embed model  : {EMBED_ENDPOINT}")

In [ ]:
# Step 1 — Ensure Change Data Feed is enabled on the source table.
# Delta Sync indexes require CDF. The ingest notebook already sets it at write time,
# but ALTER TABLE is idempotent and harmless to re-run.
spark.sql(f"""
    ALTER TABLE {CATALOG}.{SCHEMA}.{TABLE}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")
print(f"\u2705 Change Data Feed enabled on {CATALOG}.{SCHEMA}.{TABLE}")

In [ ]:
# Step 2 — Create (or reuse) the Vector Search endpoint.
from databricks.vector_search.client import VectorSearchClient

client = VectorSearchClient(disable_notice=True)

existing = [ep["name"] for ep in client.list_endpoints().get("endpoints", [])]
if VS_ENDPOINT in existing:
    print(f"\u2139\ufe0f  Endpoint already exists: {VS_ENDPOINT}")
else:
    print(f"Creating endpoint {VS_ENDPOINT} ...")
    client.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"\u2705 Endpoint created: {VS_ENDPOINT}")

# Wait until endpoint is ONLINE
import time
for attempt in range(30):
    ep = client.get_endpoint(VS_ENDPOINT)
    state = ep.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"  Endpoint state: {state}")
    if state == "ONLINE":
        break
    time.sleep(10)
else:
    raise RuntimeError(f"Endpoint {VS_ENDPOINT} did not reach ONLINE after 5 minutes")

print(f"\u2705 Endpoint ONLINE: {VS_ENDPOINT}")

In [ ]:
# Step 3 — Create (or reuse) the Delta Sync index.
existing_indexes = [
    idx["name"]
    for idx in client.list_indexes(VS_ENDPOINT).get("vector_indexes", [])
]

if VS_INDEX in existing_indexes:
    print(f"\u2139\ufe0f  Index already exists: {VS_INDEX}")
    print("  To rebuild from scratch, delete and re-run:")
    print(f"    client.delete_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)")
else:
    print(f"Creating Delta Sync index {VS_INDEX} ...")
    client.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=VS_INDEX,
        source_table_name=f"{CATALOG}.{SCHEMA}.{TABLE}",
        pipeline_type="TRIGGERED",        # manual sync trigger; use CONTINUOUS for auto-sync
        primary_key="scheme_id",
        embedding_source_column="text",   # the rich concatenated field from ingest
        embedding_model_endpoint_name=EMBED_ENDPOINT,
    )
    print(f"\u2705 Index created: {VS_INDEX}")

In [ ]:
# Step 4 — Trigger an initial sync and wait for ONLINE status.
import time

index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)

# Trigger sync (no-op if CONTINUOUS pipeline)
try:
    index.sync()
    print("Sync triggered — waiting for index to come ONLINE...")
except Exception as e:
    print(f"Sync trigger note: {e}")

for attempt in range(60):
    desc = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)
    status = desc.get("status", {})
    ready  = status.get("detailed_state", status.get("ready_for_query", "UNKNOWN"))
    print(f"  [{attempt+1:02d}] Index state: {ready}")
    if str(ready).upper() in ("ONLINE", "ONLINE_NO_PENDING_UPDATE", "TRUE"):
        break
    time.sleep(10)
else:
    raise RuntimeError("Index did not reach ONLINE state after 10 minutes")

print(f"\u2705 Index ONLINE: {VS_INDEX}")

In [ ]:
# Step 5 — Smoke test: run a sample query against the live index.
index = client.get_index(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX)

test_queries = [
    "SC female student from Maharashtra studying Class 12 with income below 2 lakh",
    "OBC student with disability pursuing undergraduate degree",
    "Muslim minority girl Class 10 income below 1 lakh",
    "EWS general category postgraduate scholarship Delhi",
    "PhD fellowship for SC girls university research",
]

print("Smoke test queries:\n")
for q in test_queries:
    results = index.similarity_search(
        query_text=q,
        columns=["scheme_id", "scheme_name"],
        num_results=3,
    )
    top = results.get("result", {}).get("data_array", [])
    names = [r[1] for r in top]
    print(f"  Q: {q[:60]}")
    print(f"  A: {names}")
    print()

print("\u2705 Vector Search index setup complete!")
print(f"\nEndpoint : {VS_ENDPOINT}")
print(f"Index    : {VS_INDEX}")
print("\nThe Databricks App will use these automatically via VS_ENDPOINT_NAME / VS_INDEX_NAME env vars.")